# 23 - Robot Dataset Engineering

## What / Why / How

**What we are trying to do:** Create a small robot episode dataset with observations, actions, instructions, statistics, and splits.

**Why this matters:** Robot learning is often bottlenecked by data quality. Bad timestamps, normalization, or splits can ruin a policy.

**How we will do it:** Generate toy episodes, compute normalization statistics, create train/validation splits, and draft a dataset card.

## Goal

Robot learning quality often depends more on data than model architecture.

You will build a tiny episode dataset with:

- Observations
- Actions
- Language instructions
- Normalization statistics
- Train/validation split

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(23)
episodes = []
for ep in range(12):
    instruction = "move to target"
    target = rng.uniform(-1, 1, size=2)
    state = rng.uniform(-1, 1, size=2)
    observations, actions = [], []
    for _ in range(30):
        action = np.clip(0.25 * (target - state), -0.12, 0.12)
        observations.append(np.r_[state, target])
        actions.append(action)
        state = state + action + rng.normal(0, 0.005, size=2)
    episodes.append({
        "instruction": instruction,
        "observations": np.array(observations),
        "actions": np.array(actions),
    })

all_obs = np.vstack([ep["observations"] for ep in episodes])
all_actions = np.vstack([ep["actions"] for ep in episodes])
stats = {
    "obs_mean": all_obs.mean(axis=0),
    "obs_std": all_obs.std(axis=0) + 1e-6,
    "action_mean": all_actions.mean(axis=0),
    "action_std": all_actions.std(axis=0) + 1e-6,
}
print("episodes:", len(episodes))
print("obs shape:", all_obs.shape, "action shape:", all_actions.shape)
print(stats)

In [ ]:
rng.shuffle(episodes)
train = episodes[:9]
val = episodes[9:]

def normalize_obs(obs):
    return (obs - stats["obs_mean"]) / stats["obs_std"]

train_obs = normalize_obs(np.vstack([ep["observations"] for ep in train]))
val_obs = normalize_obs(np.vstack([ep["observations"] for ep in val]))
print("train normalized mean:", train_obs.mean(axis=0))
print("val normalized mean:", val_obs.mean(axis=0))

In [ ]:
dataset_card = {
    "task": "2D target reaching",
    "num_episodes": len(episodes),
    "control_rate_hz": 20,
    "observation_keys": ["state_xy", "target_xy"],
    "action_keys": ["delta_xy"],
    "known_biases": ["scripted expert", "toy simulator", "no camera input"],
}
for key, value in dataset_card.items():
    print(f"{key}: {value}")

## Exercises

1. Save this dataset as `.npz` under `data/generated`.
2. Add success labels.
3. Add a data quality check for action saturation.
4. Write a dataset card for a real robot skill.